In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/m1ksj/sonar-mine-detection.git"
REPO_DIR = "/content/sonar-mine-detection"

DRIVE_DIR = "/content/drive/Shareddrives/AML_SSS_GROUP"
DATA_ZIP = f"{DRIVE_DIR}/darknet_data.zip"

RUN_NAME = "yolov4_default_baseline_t4_v1"
RUN_DIR = f"{DRIVE_DIR}/experiments/{RUN_NAME}"

print("RUN_DIR:", RUN_DIR)

!rm -rf "{RUN_DIR}"
!mkdir -p "{RUN_DIR}/checkpoints"
!mkdir -p "{RUN_DIR}/logs"
!mkdir -p "{RUN_DIR}/configs"
!mkdir -p "{RUN_DIR}/results"
!mkdir -p "{RUN_DIR}/final_model"

In [ ]:
%cd /content

!rm -rf "{REPO_DIR}"
!git clone --branch dev "{REPO_URL}" "{REPO_DIR}"

%cd "{REPO_DIR}"

!git rev-parse HEAD | tee "{RUN_DIR}/logs/repo_commit.txt"

!mkdir -p data/processed
!rm -rf data/processed/darknet
!unzip -q "{DATA_ZIP}" -d data/processed

print("Train images:")
!find data/processed/darknet/train -name "*.jpg" | wc -l

print("Val images:")
!find data/processed/darknet/val -name "*.jpg" | wc -l

print("Test images:")
!find data/processed/darknet/test -name "*.jpg" | wc -l

In [ ]:
%cd "{REPO_DIR}"

!mkdir -p models/checkpoints
!rm -rf models/checkpoints/yolov4
!ln -s "{RUN_DIR}/checkpoints" models/checkpoints/yolov4

!ls -lh models/checkpoints
!ls -lh models/checkpoints/yolov4

In [ ]:
%cd /content

!rm -rf darknet
!git clone https://github.com/AlexeyAB/darknet.git

%cd /content/darknet

!apt-get update -qq
!apt-get install -y -qq libopencv-dev

!make clean
!make -j$(nproc) GPU=1 CUDNN=1 CUDNN_HALF=1 OPENCV=1 ARCH="-gencode arch=compute_75,code=sm_75"

!ls -lh /content/darknet/darknet

In [ ]:
%cd /content/darknet

!wget -nc https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v3_optimal/yolov4.conv.137

!ls -lh yolov4.conv.137

In [ ]:
from pathlib import Path
import re

cfg_path = Path("/content/darknet/cfg/yolov4-custom.cfg")
out_path = Path("/content/darknet/cfg/yolov4-sonar-default.cfg")

lines = cfg_path.read_text().splitlines()

settings = {
    "width": "640",
    "height": "640",
    "batch": "64",
    "subdivisions": "64",
    "max_batches": "6000",
    "steps": "4800,5400",
    "saveweights": "1000",
    "savelast": "100",
}

for key, value in settings.items():
    found = False
    for i, line in enumerate(lines):
        if re.match(rf"^\s*{key}\s*=", line):
            lines[i] = f"{key}={value}"
            found = True

    if not found:
        for i, line in enumerate(lines):
            if line.strip() == "[net]":
                lines.insert(i + 1, f"{key}={value}")
                break

for i, line in enumerate(lines):
    if re.match(r"^\s*classes\s*=", line):
        lines[i] = "classes=2"

for i, line in enumerate(lines):
    if line.strip() == "[yolo]":
        j = i - 1
        while j >= 0:
            if re.match(r"^\s*filters\s*=", lines[j]):
                lines[j] = "filters=21"
                break
            j -= 1

out_path.write_text("\n".join(lines) + "\n")

print("Created:", out_path)

!grep -n "width=\|height=\|batch=\|subdivisions=\|max_batches\|steps=\|mosaic=\|jitter=\|random=\|saturation=\|exposure=\|hue=\|classes=\|filters=" /content/darknet/cfg/yolov4-sonar-default.cfg | tail -100

In [ ]:
%cd "{REPO_DIR}"

!cp /content/darknet/cfg/yolov4-sonar-default.cfg "{RUN_DIR}/configs/yolov4-sonar-default.cfg"
!cp configs/yolov4/obj.data "{RUN_DIR}/configs/obj.data"
!cp configs/yolov4/obj.names "{RUN_DIR}/configs/obj.names"
!cp configs/yolov4/train.txt "{RUN_DIR}/configs/train.txt"
!cp configs/yolov4/valid.txt "{RUN_DIR}/configs/valid.txt"
!cp configs/yolov4/test.txt "{RUN_DIR}/configs/test.txt"

!head -3 configs/yolov4/train.txt
!head -3 configs/yolov4/valid.txt
!cat configs/yolov4/obj.data

!first_image=$(head -1 configs/yolov4/train.txt) && test -f "$first_image" && echo "First train image exists"
!test -f /content/darknet/darknet && echo "Darknet binary exists"
!test -f /content/darknet/yolov4.conv.137 && echo "Pretrained weights exist"
!test -f /content/darknet/cfg/yolov4-sonar-default.cfg && echo "YOLOv4 config exists"

!nvidia-smi | tee "{RUN_DIR}/logs/nvidia_smi_before_training.txt"

In [ ]:
%cd "{REPO_DIR}"

!rm -f "{RUN_DIR}/logs/yolov4_train_log.txt"
!rm -f "{RUN_DIR}/checkpoints/"*.weights

!stdbuf -oL -eL /content/darknet/darknet detector train configs/yolov4/obj.data /content/darknet/cfg/yolov4-sonar-default.cfg /content/darknet/yolov4.conv.137 -dont_show -map 2>&1 | tee "{RUN_DIR}/logs/yolov4_train_log.txt"

In [ ]:
%cd "{REPO_DIR}"

!stdbuf -oL -eL /content/darknet/darknet detector map configs/yolov4/obj.data /content/darknet/cfg/yolov4-sonar-default.cfg models/checkpoints/yolov4/yolov4-sonar-default_best.weights -iou_thresh 0.50 2>&1 | tee "{RUN_DIR}/results/yolov4_val_map_iou50_best.txt"

!tail -120 "{RUN_DIR}/results/yolov4_val_map_iou50_best.txt"

In [ ]:
%cd "{REPO_DIR}"

test_data = """classes = 2
train = configs/yolov4/train.txt
valid = configs/yolov4/test.txt
names = configs/yolov4/obj.names
backup = models/checkpoints/yolov4/
"""

with open("configs/yolov4/obj_test.data", "w", encoding="utf-8") as file:
    file.write(test_data)

!cp configs/yolov4/obj_test.data "{RUN_DIR}/configs/obj_test.data"

!stdbuf -oL -eL /content/darknet/darknet detector map configs/yolov4/obj_test.data /content/darknet/cfg/yolov4-sonar-default.cfg models/checkpoints/yolov4/yolov4-sonar-default_best.weights -iou_thresh 0.50 2>&1 | tee "{RUN_DIR}/results/yolov4_test_map_iou50_best.txt"

!tail -120 "{RUN_DIR}/results/yolov4_test_map_iou50_best.txt"

In [ ]:
!mkdir -p "{RUN_DIR}/final_model"

!cp "{RUN_DIR}/checkpoints/yolov4-sonar-default_best.weights" "{RUN_DIR}/final_model/yolov4-sonar-default_best.weights"
!cp /content/darknet/cfg/yolov4-sonar-default.cfg "{RUN_DIR}/final_model/yolov4-sonar-default.cfg"

!ls -lh "{RUN_DIR}/final_model"

In [ ]:
import pandas as pd
from pathlib import Path

summary = pd.DataFrame([
    {
        "model": "YOLOv4 Darknet default baseline",
        "split": "test",
        "map50": 0.713504,
        "map50_percent": 71.35,
        "precision_conf_0p25": 0.71,
        "recall_conf_0p25": 0.71,
        "f1_conf_0p25": 0.71,
        "average_iou_percent": 55.69,
        "milco_ap_percent": 65.46,
        "nombo_ap_percent": 77.25,
        "tp": 72,
        "fp": 29,
        "fn": 29,
        "weights": "final_model/yolov4-sonar-default_best.weights",
        "config": "final_model/yolov4-sonar-default.cfg",
    }
])

out = Path(RUN_DIR) / "results" / "yolov4_test_summary.csv"
summary.to_csv(out, index=False)

print("Saved:", out)
display(summary)